# Replacing double line breaks with single line breaks AND  Replacing double spaces with single spaces.
REMOVE ALL SPECIAL CHARACTER  AND NON ASCII CHARACTER CODE FOR IT 


In [ ]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('C:\\Users\\taslim.siddiqui\\Downloads\\duplicate.xlsx')

# Specify the original column name
original_column = 'concepts'
new_column = 'concepts_cleaned'

# Function to remove duplicates within a cell
def remove_duplicates_in_cell(cell):
    if pd.isna(cell):
        return cell
    items = [item.strip() for item in str(cell).split(',')]
    unique_items = list(dict.fromkeys(items))  # Preserves order
    return ', '.join(unique_items)

# Create a new column with cleaned values
df[new_column] = df[original_column].apply(remove_duplicates_in_cell)

# Save the file with both columns
df.to_excel('C:\\Users\\taslim.siddiqui\\Downloads\\cleaned_file1.xlsx', index=False)


In [16]:
import pandas as pd
import re

# Use exact column name with trailing space
columns_to_clean = ["Syllabus "]  

def clean_text(text):
    if pd.notnull(text):
        text = str(text)
        text = re.sub(r'[\|\_!•?—\-*]+', ' ', text)
        text = re.sub(r'\s{2,}', ' ', text)
        text = text.encode('ascii', 'ignore').decode('ascii')
        text = text.strip()
        return text
    return text

try:
    input_file_path = r"C:\Users\taslim.siddiqui\Downloads\edx_course_QC_Collation file 1 1.xlsx"
    df = pd.read_excel(input_file_path, sheet_name="Sheet2")

    print("Columns found in the file:")
    print(df.columns.tolist())

    for col in columns_to_clean:
        if col in df.columns:
            print(f"Cleaning column → {col}")
            df[col] = df[col].apply(clean_text)
        else:
            print(f"⚠️ Column '{col}' not found — skipped.")

    output_file_path = r"C:\Users\taslim.siddiqui\Downloads\cleaned_output_20-11-2025.xlsx"
    df.to_excel(output_file_path, index=False, engine='openpyxl')

    print(f"✅ Cleaned data saved successfully as '{output_file_path}'.")

except Exception as e:
    print(f"❌ An error occurred: {e}")


Columns found in the file:
['URL', 'Course_name', 'Course_Syllabus', 'Syllabus ', 'About course']
Cleaning column → Syllabus 
✅ Cleaned data saved successfully as 'C:\Users\taslim.siddiqui\Downloads\cleaned_output_20-11-2025.xlsx'.


In [ ]:
import re
from bs4 import BeautifulSoup
import pandas as pd

# Parse HTML content
file_path = "C:\\Users\\taslim.siddiqui\\Downloads\\response10Courses..html"  # Replace with your file path

with open(file_path, "r", encoding="utf-8") as file:
    html_content = file.read()

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Get all visible text
text = soup.get_text(strip=True)

# Split the text into lines for better structure
lines = text.splitlines()

# Initialize an empty list to hold the processed lines
processed_lines = []

# Loop through lines and process them
for line in lines:
    # Remove the word "Preview" without altering the structure
    cleaned_line = line.replace('Preview', '').strip()

    # Ensure 'Section' and the number are not split onto separate lines
   

    # Ensure numbers are separated by a newline
    cleaned_line = re.sub(r'(\d+)', r'\n\1  ', cleaned_line)

    cleaned_line= re.sub(r'(Section)\s*\n(\d)', r'\1 \2', cleaned_line)

    cleaned_line= re.sub(r"(.+?)(Section \d+)", r"\1\n\2", cleaned_line)

   

    # Add processed line if it's not empty
    if cleaned_line != '':
        processed_lines.append(cleaned_line)

# Join the processed lines into a single string
processed_text = "\n".join(processed_lines)

# Convert the processed lines into a DataFrame
df = pd.DataFrame([processed_text], columns=["Content"])

# Save to Excel
output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\output1.xlsx"
df.to_excel(output_path, index=False)

# Display the first few rows in the console for review
print(df.head(6))

# Uncomment the following line if you want a success message after saving
# print(f"Data saved to {output_path}") and add more  line of Section with  \n

# SPLIT  Row  numbers by comma delimiter  example concept_id

In [2]:
## both concept and function 

import pandas as pd

# Load your file
file_path = r"C:\Users\taslim.siddiqui\Downloads\Kotak FSC - 06.11.xlsx"
df = pd.read_excel(file_path)

# Drop rows where either 'concepts' or 'function_ids' is missing
df = df.dropna(subset=['concepts', 'function_ids'])

# Convert to string and split by comma
df['concepts'] = df['concepts'].astype(str).str.split(r'\s*,\s*')
df['function_ids'] = df['function_ids'].astype(str).str.split(r'\s*,\s*')

# Explode both columns (creates all combinations)
df = df.explode('concepts')
df = df.explode('function_ids').reset_index(drop=True)

# Remove blank or whitespace-only values
df = df[df['concepts'].str.strip().astype(bool)]
df = df[df['function_ids'].str.strip().astype(bool)]

# Reorder to show concepts and function_ids first
cols = ['concepts', 'function_ids'] + [col for col in df.columns if col not in ['concepts', 'function_ids']]
df = df[cols]

# Save to Excel
output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\Split_Concepts_Functions_06.xlsx"
df.to_excel(output_path, index=False)

print(f"✅ File saved to: {output_path}")




✅ File saved to: C:\Users\taslim.siddiqui\Downloads\Split_Concepts_Functions_06.xlsx


In [ ]:
import pandas as pd 
import requests
from bs4 import BeautifulSoup
 
def scrape_pharmastate_course(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')

    # Course Name
    course_name_div = soup.find('div', class_='elementor-widget-etlms-course-title')
    course_name = course_name_div.get_text(strip=True) if course_name_div else ''

    # ✅ Description: Clean and truncate
    desc_div = soup.find('div', class_='tutor-single-course-segment etlms-course-description')
    full_desc = desc_div.get_text(separator=" ", strip=True) if desc_div else ''

    # Truncate after "Why should you attend?" or "What will you learn?"
    cut_keywords = ["Why should you attend?", "What will you learn?"]
    cut_index = min([full_desc.find(k) for k in cut_keywords if k in full_desc] or [len(full_desc)])
    description = full_desc[:cut_index].strip()

    # ✅ Remove "Description" from start
    if description.lower().startswith("description"):
        description = description[len("description"):].strip(": ").strip()

    # Selling Price
    offer_price_tag = soup.find('del')
    selling_price = offer_price_tag.get_text(strip=True).replace("₹", "").strip() if offer_price_tag else ''

    # Offer Price
    price_span = soup.find('span', class_='woocommerce-Price-amount')
    offer_price = price_span.get_text(strip=True).replace("₹", "").strip() if price_span else ''

    # Duration
    duration_span = soup.find('span', class_='tutor-meta-value tutor-color-secondary tutor-mr-4')
    duration = duration_span.get_text(strip=True) if duration_span else ''

    # Course Level
    course_level_span = soup.find('span', class_='tutor-fs-7 tutor-fw-medium tutor-color-black etlms-enrolled-label-value')
    course_level = course_level_span.get_text(strip=True) if course_level_span else ''
    if course_level.lower() == "all levels":
        course_level = "Advanced"

    # Duration Time
    duration_parts = soup.find_all('span', class_='tutor-meta-level')
    if len(duration_parts) >= 2:
        hour = duration_parts[0].get_text(strip=True).zfill(2)
        minute = duration_parts[1].get_text(strip=True).zfill(2)
        duration_time = f"{hour}:{minute}"
    else:
        duration_time = ''

    # What I will learn
    what_learn = ''
    for h3 in soup.find_all('h3'):
        if 'What I will learn' in h3.get_text():
            ul = h3.find_next_sibling('ul')
            if ul:
                what_learn = ' \n '.join(li.get_text(strip=True) for li in ul.find_all('li'))

    # Who should take it
    who_should_take_it = ''
    for h3 in soup.find_all('h3'):
        if 'Target Audience' in h3.get_text():
            ul = h3.find_next_sibling('ul')
            if ul:
                who_should_take_it = ' \n '.join(li.get_text(strip=True) for li in ul.find_all('li'))

    # Educator
    educator_tag = soup.find('a', class_='tutor-instructor-name tutor-fs-6 tutor-fw-bold tutor-color-black')
    educator = educator_tag.get_text(strip=True) if educator_tag else ''

    # Fee Structure
    fee_structure = (
        f"{offer_price} \n -All other fees remain unchanged\n"
        "-Education loans are available through leading banks and NBFCs.\n"
        "-EMI options are also available for your convenience."
    ) if offer_price else (
        "-All other fees remain unchanged\n"
        "-Education loans are available through leading banks and NBFCs.\n"
        "-EMI options are also available for your convenience."
    )

    return {
        'Course Name': course_name,
        'Description': description,
        'Course Link': url,
        'Selling Price': selling_price,
        'Offer Price': offer_price,
        'Duration': duration,
        'Duration Time': duration_time,
        'Fee Structure': fee_structure,
        'Course Level': course_level,
        'Educator': educator,
        'Who should take it': who_should_take_it,
        'What I will learn': what_learn
    }

# Run test on 1 course
if __name__ == "__main__":
    url = "https://pharmastate.academy/courses/masterclass-trends-insights-and-patient-centric-strategies-by-nivedita-parulekar/"
    data = scrape_pharmastate_course(url)
    df = pd.DataFrame([data])
    df.to_excel("C:\\Users\\taslim.siddiqui\\Downloads\\pharmastate_course_data.xlsx", index=False)
    print("✅ Scraped and saved to Excel (description cleaned).")


# #web **Srapping**

# WEB SCARPPING PHARMASTATE

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

def scrape_pharmastate_course(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')

    # Course Name
    course_name_div = soup.find('div', class_='elementor-widget-etlms-course-title')
    course_name = course_name_div.get_text(strip=True) if course_name_div else ''

    # Description
    desc_div = soup.find('div', class_='tutor-single-course-segment etlms-course-description')
    description = ''
    if desc_div:
        raw_text = desc_div.get_text(separator=' ', strip=True)
        stop_keywords = [
            'what will you learn',
            'why should you attend',
            'What will your learn'
            'who should enroll',
            'What will I learn ',
            'What will your learn',
            'What will I learn',
            'language:',
            'objective of sop',
            'how to download course certificate'
        ]
        stop_index = len(raw_text)
        for keyword in stop_keywords:
            idx = raw_text.lower().find(keyword)
            if idx != -1 and idx < stop_index:
                stop_index = idx
        description = raw_text[:stop_index].strip()
        if description.lower().startswith("description"):
            description = description[len("description"):].strip(": ").strip()

    # What I will learn
    what_learn = ''
    learn_ul = soup.find('ul', class_='etlms-course-widget-list-items')
    if learn_ul:
        items = learn_ul.find_all('span', class_='tutor-list-label')
        what_learn = '\n'.join(item.get_text(strip=True) for item in items)

    if not what_learn and desc_div:
        for tag in desc_div.find_all(['strong', 'h3', 'h2', 'b', 'p']):
            if 'what will you learn' in tag.get_text(strip=True).lower():
                next_node = tag.find_next_sibling()
                result = []
                while next_node and next_node.name in ['ul', 'p']:
                    text = next_node.get_text(strip=True)
                    if text:
                        result.append(text)
                    next_node = next_node.find_next_sibling()
                what_learn = '\n'.join(result)
                break

    for phrase in ['Language: English', 'How to Download Course certificate: Watch Video']:
        what_learn = what_learn.replace(phrase, '').strip()

    # Prices
    price_spans = soup.find_all('span', class_='woocommerce-Price-amount')
    prices = [span.find('bdi').get_text(strip=True).replace("₹", "").replace(",", "").strip()
              for span in price_spans if span.find('bdi')]
    selling_price, offer_price = '', ''
    if len(prices) >= 2:
        selling_price, offer_price = prices[0], prices[1]
    elif len(prices) == 1:
        offer_price = prices[0]
        selling_price = prices[0]

    if offer_price in ['0', '0.00', '', None] and selling_price:
        offer_price = selling_price

    # Duration (text)
    duration_span = soup.find('span', class_='tutor-meta-value tutor-color-secondary tutor-mr-4')
    duration = duration_span.get_text(strip=True) if duration_span else ''
    duration = duration.replace("hours", "Hours").replace("minutes", "Minutes")

    # Duration Time and Rounded Duration
    parts = soup.find_all('span', class_='tutor-meta-level')
    if len(parts) >= 2:
        h = int(parts[0].get_text(strip=True))
        m = int(parts[1].get_text(strip=True))
        duration_time = f"{h:02d}:{m:02d}"
        if m < 15:
            rounded = str(h)
        elif m < 45:
            rounded = str(h + 0.5)
        else:
            rounded = str(h + 1)
    elif len(parts) == 1:
        h, m = 0, int(parts[0].get_text(strip=True))
        duration_time = str(m)
        rounded = str(m)
    else:
        duration_time = ''
        rounded = ''

    # Fee Structure
    fee_structure = (
        f"{offer_price} \n -All other fees remain unchanged\n"
        "-Education loans are available through leading banks and NBFCs.\n"
        "-EMI options are also available for your convenience."
    ) if offer_price else (
        "-All other fees remain unchanged\n"
        "-Education loans are available through leading banks and NBFCs.\n"
        "-EMI options are also available for your convenience."
    )

    # Language
    language = ''
    for p in soup.find_all('p'):
        if 'Language:' in p.get_text():
            language = p.get_text().replace("Language:", "").strip()
            break

    # Course Level
    course_level_span = soup.find('span', class_='tutor-fs-7 tutor-fw-medium tutor-color-black etlms-enrolled-label-value')
    course_level = course_level_span.get_text(strip=True) if course_level_span else ''
    if course_level.lower() == "all levels":
        course_level = "Beginner"

    # Certificate
    certificate = ''
    spans = soup.find_all('span', class_='tutor-fs-7 tutor-fw-medium tutor-color-black etlms-enrolled-label-value')
    for span in spans:
        text = span.get_text(strip=True)
        if 'certificate' in text.lower():
            certificate = text
            break

    # Certificate Image
    cert_img_tag = soup.find('img', alt='selected template')
    certificate_image = cert_img_tag['src'] if cert_img_tag and 'src' in cert_img_tag.attrs else ''

    # Who should take it
    who_should_take_it = ''
    for h3 in soup.find_all('h3'):
        if 'Target Audience' in h3.get_text():
            ul = h3.find_next_sibling('ul')
            if ul:
                who_should_take_it = ' \n '.join(li.get_text(strip=True) for li in ul.find_all('li'))

    # Educator
    educator_tags = soup.find_all('a', class_='tutor-instructor-name tutor-fs-6 tutor-fw-bold tutor-color-black')
    educators = [tag.get_text(strip=True) for tag in educator_tags]
    educators = [e for e in educators if e.lower() != "pharmastate academy"]
    educator = ', '.join(educators)

    return {
        'Course Name': course_name,
        'Course Link': url,
        'Description': description,
        'Selling Price': selling_price,
        'Offer Price': offer_price,
        'Duration': duration,
        'Duration Time': duration_time,
        'Rounded Duration': rounded,
        'Fee Structure': fee_structure,
        'Language': language,
        'Course Level': course_level,
        'Certificate': certificate,
        'Certificate Image': certificate_image,
        'Educator': educator,
        'Who should take it': who_should_take_it,
        'What I will learn': what_learn
    }

# =======================
# Run for All Courses
# =======================
if __name__ == "__main__":
    input_path = "C:\\Users\\taslim.siddiqui\\Downloads\\pharmastate (1).xlsx"
    output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\pharmastate_course_Data_ALL _PART2.xlsx"

    df_links = pd.read_excel(input_path)
    urls = df_links['Course Link'].dropna().unique()

    all_data = []
    for i, url in enumerate(urls):
        print(f"Scraping ({i+1}/{len(urls)}): {url}")
        try:
            data = scrape_pharmastate_course(url)
        except Exception as e:
            print(f"❌ Error scraping {url}: {e}")
            data = {col: '' for col in [
                'Course Name', 'Course Link', 'Description', 'Selling Price',
                'Offer Price', 'Duration', 'Duration Time', 'Rounded Duration', 'Fee Structure', 'Language',
                'Course Level', 'Certificate', 'Certificate Image', 'Educator',
                'Who should take it', 'What I will learn'
            ]}
            data['Course Link'] = url
        all_data.append(data)
        time.sleep(1)

    df_out = pd.DataFrame(all_data)
    df_out.to_excel(output_path, index=False)
    print("✅ All courses saved to:", output_path)


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

def scrape_pharmastate_course(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')

    # Course Name
    course_name_div = soup.find('div', class_='elementor-widget-etlms-course-title')
    course_name = course_name_div.get_text(strip=True) if course_name_div else ''

    # Description
    desc_div = soup.find('div', class_='tutor-single-course-segment etlms-course-description')
    description = ''
    if desc_div:
        raw_text = desc_div.get_text(separator=' ', strip=True)
        stop_keywords = [
            'what will you learn',
            'why should you attend',
            'What will your learn'
            'who should enroll',
            'What will I learn ',
            'What will your learn',
            'What will I learn',
            'language:',
            'objective of sop',
            'how to download course certificate'
        ]
        stop_index = len(raw_text)
        for keyword in stop_keywords:
            idx = raw_text.lower().find(keyword)
            if idx != -1 and idx < stop_index:
                stop_index = idx
        description = raw_text[:stop_index].strip()
        if description.lower().startswith("description"):
            description = description[len("description"):].strip(": ").strip()

    # What I will learn
    what_learn = ''
    learn_ul = soup.find('ul', class_='etlms-course-widget-list-items')
    if learn_ul:
        items = learn_ul.find_all('span', class_='tutor-list-label')
        what_learn = '\n'.join(item.get_text(strip=True) for item in items)

    if not what_learn and desc_div:
        for tag in desc_div.find_all(['strong', 'h3', 'h2', 'b', 'p']):
            if 'what will you learn' in tag.get_text(strip=True).lower():
                next_node = tag.find_next_sibling()
                result = []
                while next_node and next_node.name in ['ul', 'p']:
                    text = next_node.get_text(strip=True)
                    if text:
                        result.append(text)
                    next_node = next_node.find_next_sibling()
                what_learn = '\n'.join(result)
                break

    for phrase in ['Language: English', 'How to Download Course certificate: Watch Video']:
        what_learn = what_learn.replace(phrase, '').strip()

    # Prices
    price_spans = soup.find_all('span', class_='woocommerce-Price-amount')
    prices = [span.find('bdi').get_text(strip=True).replace("₹", "").replace(",", "").strip()
              for span in price_spans if span.find('bdi')]
    selling_price, offer_price = '', ''
    if len(prices) >= 2:
        selling_price, offer_price = prices[0], prices[1]
    elif len(prices) == 1:
        offer_price = prices[0]
        selling_price = prices[0]

    if offer_price in ['0', '0.00', '', None] and selling_price:
        offer_price = selling_price

    # Duration (text)
    duration_span = soup.find('span', class_='tutor-meta-value tutor-color-secondary tutor-mr-4')
    duration = duration_span.get_text(strip=True) if duration_span else ''
    duration = duration.replace("hours", "Hours").replace("minutes", "Minutes")

    # Duration Time and Rounded Duration
    parts = soup.find_all('span', class_='tutor-meta-level')
    if len(parts) >= 2:
        h = int(parts[0].get_text(strip=True))
        m = int(parts[1].get_text(strip=True))
        duration_time = f"{h:02d}:{m:02d}"
        if m < 15:
            rounded = str(h)
        elif m < 45:
            rounded = str(h + 0.5)
        else:
            rounded = str(h + 1)
    elif len(parts) == 1:
        h, m = 0, int(parts[0].get_text(strip=True))
        duration_time = str(m)
        rounded = str(m)
    else:
        duration_time = ''
        rounded = ''

    # Fee Structure
    fee_structure = (
        f"{offer_price} \n -All other fees remain unchanged\n"
        "-Education loans are available through leading banks and NBFCs.\n"
        "-EMI options are also available for your convenience."
    ) if offer_price else (
        "-All other fees remain unchanged\n"
        "-Education loans are available through leading banks and NBFCs.\n"
        "-EMI options are also available for your convenience."
    )

    # Language
    language = ''
    for p in soup.find_all('p'):
        if 'Language:' in p.get_text():
            language = p.get_text().replace("Language:", "").strip()
            break

    # Course Level
    course_level_span = soup.find('span', class_='tutor-fs-7 tutor-fw-medium tutor-color-black etlms-enrolled-label-value')
    course_level = course_level_span.get_text(strip=True) if course_level_span else ''
    if course_level.lower() == "all levels":
        course_level = "Beginner"

    # Certificate
    certificate = ''
    spans = soup.find_all('span', class_='tutor-fs-7 tutor-fw-medium tutor-color-black etlms-enrolled-label-value')
    for span in spans:
        text = span.get_text(strip=True)
        if 'certificate' in text.lower():
            certificate = text
            break

    # Certificate Image
    cert_img_tag = soup.find('img', alt='selected template')
    certificate_image = cert_img_tag['src'] if cert_img_tag and 'src' in cert_img_tag.attrs else ''

    # Who should take it
    who_should_take_it = ''
    for h3 in soup.find_all('h3'):
        if 'Target Audience' in h3.get_text():
            ul = h3.find_next_sibling('ul')
            if ul:
                who_should_take_it = ' \n '.join(li.get_text(strip=True) for li in ul.find_all('li'))

    # Educator
    educator_tags = soup.find_all('a', class_='tutor-instructor-name tutor-fs-6 tutor-fw-bold tutor-color-black')
    educators = [tag.get_text(strip=True) for tag in educator_tags]
    educators = [e for e in educators if e.lower() != "pharmastate academy"]
    educator = ', '.join(educators)

    return {
        'Course Name': course_name,
        'Course Link': url,
        'Description': description,
        'Selling Price': selling_price,
        'Offer Price': offer_price,
        'Duration': duration,
        'Duration Time': duration_time,
        'Rounded Duration': rounded,
        'Fee Structure': fee_structure,
        'Language': language,
        'Course Level': course_level,
        'Certificate': certificate,
        'Certificate Image': certificate_image,
        'Educator': educator,
        'Who should take it': who_should_take_it,
        'What I will learn': what_learn
    }

# =======================
# Run for All Courses
# =======================
if __name__ == "__main__":
    input_path = "C:\\Users\\taslim.siddiqui\\Downloads\\pharmastate (1).xlsx"
    output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\pharmastate_course_Data_ALL _PART2.xlsx"

    df_links = pd.read_excel(input_path)
    urls = df_links['Course Link'].dropna().unique()

    all_data = []
    for i, url in enumerate(urls):
        print(f"Scraping ({i+1}/{len(urls)}): {url}")
        try:
            data = scrape_pharmastate_course(url)
        except Exception as e:
            print(f"❌ Error scraping {url}: {e}")
            data = {col: '' for col in [
                'Course Name', 'Course Link', 'Description', 'Selling Price',
                'Offer Price', 'Duration', 'Duration Time', 'Rounded Duration', 'Fee Structure', 'Language',
                'Course Level', 'Certificate', 'Certificate Image', 'Educator',
                'Who should take it', 'What I will learn'
            ]}
            data['Course Link'] = url
        all_data.append(data)
        time.sleep(1)

    df_out = pd.DataFrame(all_data)
    df_out.to_excel(output_path, index=False)
    print("✅ All courses saved to:", output_path)


extract course syllabus for pharma state

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def clean_text(text):
    # Remove non-ASCII characters (like �, emojis)
    return re.sub(r'[^\x00-\x7F]+', '', text).strip()

def scrape_pharmastate_course_cleaned(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')

    # Course Name
    course_name_div = soup.find('div', class_='elementor-widget-etlms-course-title')
    course_name = clean_text(course_name_div.get_text(strip=True)) if course_name_div else ''

    curriculum = []

    modules = soup.find_all('div', class_='tutor-accordion-item')

    for module in modules:
        module_title_tag = module.find('h4', class_='tutor-accordion-item-header')
        module_title = clean_text(module_title_tag.get_text(strip=True)) if module_title_tag else "Untitled Module"

        lesson_tags = module.find_all('li', class_='tutor-course-content-list-item')
        lesson_list = []

        for lesson in lesson_tags:
            title = lesson.find('h5', class_='tutor-course-content-list-item-title')
            title_clean = clean_text(title.get_text(strip=True)) if title else "Untitled Lesson"

            lesson_list.append({'title': title_clean})

        curriculum.append({
            'module': module_title,
            'lessons': lesson_list
        })

    # Format curriculum string without durations or special characters
    formatted_curriculum = ""
    for module in curriculum:
        formatted_curriculum += f"{module['module']}\n"
        for lesson in module['lessons']:
            formatted_curriculum += f"   - {lesson['title']}\n"
        formatted_curriculum += "\n"

    return {
        'Course Name': course_name,
        'Course Link': url,
        'Course Curriculum': formatted_curriculum.strip()
    }

# === MAIN: Scrape from Excel file ===
if __name__ == "__main__":
    input_path = "C:\\Users\\taslim.siddiqui\\Downloads\\course link_NEW.xlsx"
    output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\pharmastate_all_courses_cleaned.xlsx"

    df_links = pd.read_excel(input_path)

    if 'Course Link' not in df_links.columns:
        raise ValueError("❌ 'Course Link' column not found in Excel file.")

    urls = df_links['Course Link'].dropna().unique()

    all_data = []
    for i, url in enumerate(urls):
        print(f"🔄 Scraping ({i+1}/{len(urls)}): {url}")
        try:
            data = scrape_pharmastate_course_cleaned(url)
        except Exception as e:
            print(f"❌ Error scraping {url}: {e}")
            data = {
                'Course Name': '',
                'Course Link': url,
                'Course Curriculum': ''
            }
        all_data.append(data)
        time.sleep(1)  # polite delay between requests

    df_out = pd.DataFrame(all_data)
    df_out.to_excel(output_path, index=False)

    print(f"\n✅ Saved all {len(all_data)} courses to Excel: {output_path}")
